# Theory Revision v4 — E4

Implementação do protocolo versionado em `4183b5e` e das decisões técnicas
congeladas em `54a65af`. Este notebook edita objetos-regra formais e imutáveis.
Ele não executa o DptOIE, não modifica Java, não usa solver e não consulta o
GOLD durante matching ou aplicação.

In [1]:
from dataclasses import dataclass, fields, is_dataclass, replace
from pathlib import Path
from typing import Any
import hashlib
import inspect
import json
import platform

import networkx as nx
import openpyxl

PROTOCOL_COMMIT = "4183b5ecddee1f8e42d82f5d21de2381513384b6"
DECISIONS_COMMIT = "54a65af0e2023eb23683f1e020123ac692401c98"
SOURCE_HEAD = DECISIONS_COMMIT

SPREADSHEET = Path("research/experimento/semana5_experimento_OIEC_PT_GOLD (3)(1).xlsx")
PARSE_PATHS = {
    "S009": Path("logs/s009/reproducao_instrumentada_trace_on/output/output_DP.conll"),
    "S003": Path("logs/e4/controles_baseline/s003/output/output_DP.conll"),
    "S005": Path("logs/e4/controles_baseline/s005/output/output_DP.conll"),
    "S010": Path("logs/e4/controles_baseline/s010/output/output_DP.conll"),
}
EXPECTED_HASHES = {
    str(SPREADSHEET): "a00c037da53f5de90307d8918b8547d226888297eb7243d0272c68d5f337616c",
    str(PARSE_PATHS["S009"]): "e94d708cef67efaa363a7fdb9e646e4fa39bc3c379e871923bf3631e96e2967f",
    str(PARSE_PATHS["S003"]): "f12731b7b6dc71ffda49d837f947f8eb76119751a4062585c8e564e3d0cb624c",
    str(PARSE_PATHS["S005"]): "49335c1eb391b21f55b6c44a956bfbfcc15ae968e75eb511f364e8a16a20ec2d",
    str(PARSE_PATHS["S010"]): "c3aaab2679596c96ca24e51896134ef572d7f963946c0af5c16ae50c4f4e5cfe",
}

TRUE_PROVISIONAL = "TRUE_PROVISIONAL"
FALSE_PROVISIONAL = "FALSE_PROVISIONAL"
INDETERMINATE = "INDETERMINATE"
REGE_STATUSES = (TRUE_PROVISIONAL, FALSE_PROVISIONAL, INDETERMINATE)

DEPENDENCY = "DEPENDENCY"
OPENIE = "OPENIE"
CONSTRUCTION = "CONSTRUCTION"
LINGUISTIC = "LINGUISTIC"

ARG_ROOT_ALLOWED = (
    "nmod", "acl:part", "xcomp", "xcomp:adj", "nummod", "advmod",
    "amod", "obj", "iobj", "acl:relcl",
)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

OBSERVED_HASHES = {path: sha256(path) for path in EXPECTED_HASHES}
assert OBSERVED_HASHES == EXPECTED_HASHES

ENVIRONMENT = {
    "python": platform.python_version(),
    "networkx": nx.__version__,
    "openpyxl": openpyxl.__version__,
    "protocol_commit": PROTOCOL_COMMIT,
    "decisions_commit": DECISIONS_COMMIT,
    "source_head": SOURCE_HEAD,
}
print(json.dumps({"environment": ENVIRONMENT, "input_hashes": OBSERVED_HASHES}, ensure_ascii=False, indent=2))

{
  "environment": {
    "python": "3.12.0",
    "networkx": "3.5",
    "openpyxl": "3.1.5",
    "protocol_commit": "4183b5ecddee1f8e42d82f5d21de2381513384b6",
    "decisions_commit": "54a65af0e2023eb23683f1e020123ac692401c98",
    "source_head": "54a65af0e2023eb23683f1e020123ac692401c98"
  },
  "input_hashes": {
    "research\\experimento\\semana5_experimento_OIEC_PT_GOLD (3)(1).xlsx": "a00c037da53f5de90307d8918b8547d226888297eb7243d0272c68d5f337616c",
    "logs\\s009\\reproducao_instrumentada_trace_on\\output\\output_DP.conll": "e94d708cef67efaa363a7fdb9e646e4fa39bc3c379e871923bf3631e96e2967f",
    "logs\\e4\\controles_baseline\\s003\\output\\output_DP.conll": "f12731b7b6dc71ffda49d837f947f8eb76119751a4062585c8e564e3d0cb624c",
    "logs\\e4\\controles_baseline\\s005\\output\\output_DP.conll": "49335c1eb391b21f55b6c44a956bfbfcc15ae968e75eb511f364e8a16a20ec2d",
    "logs\\e4\\controles_baseline\\s010\\output\\output_DP.conll": "c3aaab2679596c96ca24e51896134ef572d7f963946c0af5c16ae50c4f

## 1. Dados reais e separação entre construção e avaliação

Os tokens e as dependências são lidos dos CoNLL preservados. Os componentes
baseline são confrontados com a planilha. GOLD e estado esperado ficam em
objetos de avaliação separados e não entram nas assinaturas do matcher ou do
executor.

In [2]:
def load_conll(path):
    tokens = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        values = line.split("\t")
        token_id = int(values[0])
        tokens[token_id] = {
            "id": token_id,
            "form": values[1],
            "pos": values[3],
            "head": int(values[6]),
            "deprel": values[7],
            "ordem": token_id,
        }
    return tokens

PARSES = {sentence_id: load_conll(path) for sentence_id, path in PARSE_PATHS.items()}

workbook = openpyxl.load_workbook(SPREADSHEET, read_only=True, data_only=True)
sheet = workbook["analise"]
raw_rows = list(sheet.iter_rows(values_only=True))
headers = raw_rows[0]
ANALYSIS_ROWS = {
    f"A{excel_row}": dict(zip(headers, values))
    for excel_row, values in enumerate(raw_rows[1:], start=2)
}

@dataclass(frozen=True)
class TargetInput:
    n: int
    status_rege: str
    status_origin: str
    status_uses_gold: bool
    justification: str

@dataclass(frozen=True)
class ConstructionCase:
    case_id: str
    sentence_id: str
    row_id: str
    kind: str
    arg0_base: tuple
    rel_base: tuple
    arg1_baseline: tuple
    u: tuple
    relation_core: int
    targets: tuple
    notes: str = ""

@dataclass(frozen=True)
class EvaluationOracle:
    case_id: str
    expected_e4_arg0: tuple
    expected_e4_rel: tuple
    expected_e4_arg1: tuple
    gold_arg0: tuple
    gold_rel: tuple
    gold_arg1: tuple
    expected_p_by_n: tuple

MANUAL_ORIGIN = (
    "classificação manual provisória do protocolo v3; apoiada por sentença, "
    "GOLD, parse e justificativa gramatical; não é reconhecedor operacional"
)

def target(n, status, justification, uses_gold=True):
    assert status in REGE_STATUSES
    return TargetInput(n, status, MANUAL_ORIGIN, uses_gold, justification)

CASES = (
    ConstructionCase("S009-A41", "S009", "A41", "E_MINUS_PRIMARY", (1,), (2,), (3,4,5), tuple(), 2,
                     (target(5, TRUE_PROVISIONAL, "voltar para; proveniência confirmada em A41"),)),
    ConstructionCase("S009-A42", "S009", "A42", "E_MINUS_VARIANT", (1,), (2,), tuple(range(3,11)), tuple(range(6,11)), 2,
                     (target(5, TRUE_PROVISIONAL, "mesmo par voltar–para da sentença S009"),)),
    ConstructionCase("S009-A43", "S009", "A43", "E_MINUS_MIXED", (1,), (2,), tuple(range(3,20)), tuple(range(6,20)), 2,
                     (target(5, TRUE_PROVISIONAL, "mesmo par voltar–para; E3/S3 fora do alvo"),), "E4+E3+S3"),
    ConstructionCase("S009-A44", "S009", "A44", "E_MINUS_MIXED", (1,), (2,), tuple(range(3,26)), tuple(range(6,26)), 2,
                     (target(5, TRUE_PROVISIONAL, "mesmo par voltar–para; E3/S3 fora do alvo"),), "E4+E3+S3"),
    ConstructionCase("S005-A28", "S005", "A28", "CONTROL", (15,), (16,), tuple(range(17,25)), (17,20,21), 16,
                     (target(19, FALSE_PROVISIONAL, "para marca continuar; não regência de tem"),
                      target(24, FALSE_PROVISIONAL, "contra pertence ao predicado encaixado protestar"))),
    ConstructionCase("S010-A51", "S010", "A51", "CONTROL", (1,), (2,), tuple(range(3,11)), (3,4,5,9,10), 2,
                     (target(8, FALSE_PROVISIONAL, "locativo na Universidade; não regência de visitei"),),
                     "fronteiras GOLD iguais; superfície en a/em a"),
    ConstructionCase("S010-A52", "S010", "A52", "CONTROL", (1,), (2,), tuple(range(3,15)), (3,4,5,9,10,13,14), 2,
                     (target(8, FALSE_PROVISIONAL, "locativo na Universidade; não regência de visitei"),
                      target(12, FALSE_PROVISIONAL, "para marca oração de finalidade")),
                     "fronteiras GOLD iguais; superfície en a/em a"),
    ConstructionCase("S003-A20", "S003", "A20", "CONTROL", (7,), (8,), tuple(range(9,15)), (9,10,11,12), 8,
                     (target(14, FALSE_PROVISIONAL, "de atenção complementa digno; não a cópula"),),
                     "parse global discutível"),
)

ORACLES = {
    "S009-A41": EvaluationOracle("S009-A41", (1,), (2,3), (4,5), (1,), (2,3), (4,5), ((5,3),)),
    "S009-A42": EvaluationOracle("S009-A42", (1,), (2,3), tuple(range(4,11)), (1,), (2,3), tuple(range(4,11)), ((5,3),)),
    "S009-A43": EvaluationOracle("S009-A43", (1,), (2,3), tuple(range(4,20)), (1,), (2,3), tuple(range(4,11)), ((5,3),)),
    "S009-A44": EvaluationOracle("S009-A44", (1,), (2,3), tuple(range(4,26)), (1,), (2,3), tuple(range(4,11)), ((5,3),)),
    "S005-A28": EvaluationOracle("S005-A28", (15,), (16,), tuple(range(17,25)), (15,), (16,), tuple(range(17,25)), ((19,18),(24,22))),
    "S010-A51": EvaluationOracle("S010-A51", (1,), (2,), tuple(range(3,11)), (1,), (2,), tuple(range(3,11)), ((8,6),)),
    "S010-A52": EvaluationOracle("S010-A52", (1,), (2,), tuple(range(3,15)), (1,), (2,), tuple(range(3,15)), ((8,6),(12,11))),
    "S003-A20": EvaluationOracle("S003-A20", (7,), (8,), tuple(range(9,15)), (7,), (8,), tuple(range(9,15)), ((14,13),)),
}

def token_text(sentence_id, token_ids):
    return " ".join(PARSES[sentence_id][token_id]["form"] for token_id in token_ids)

def normalize_space(value):
    return " ".join(str(value).split())

for case in CASES:
    row = ANALYSIS_ROWS[case.row_id]
    assert row["id_sentenca"] == case.sentence_id
    assert row["extrator"] == "DPTOIE"
    assert normalize_space(token_text(case.sentence_id, case.arg0_base)) == normalize_space(row["arg0_extraido"])
    assert normalize_space(token_text(case.sentence_id, case.rel_base)) == normalize_space(row["rel_extraida"])
    assert normalize_space(token_text(case.sentence_id, case.arg1_baseline)) == normalize_space(row["arg1_extraido"])

print("REAL CASES", [case.case_id for case in CASES])

REAL CASES ['S009-A41', 'S009-A42', 'S009-A43', 'S009-A44', 'S005-A28', 'S010-A51', 'S010-A52', 'S003-A20']


## 2. Álgebra imutável de regras

A representação distingue contexto, binders, guardas, efeitos, restrições,
proveniência e linhagem. Expressões de conjuntos de tokens são avaliadas pelo
executor a partir do match.

In [3]:
@dataclass(frozen=True, order=True)
class Variable:
    name: str
    type: str

@dataclass(frozen=True, order=True)
class ContextAtom:
    expression: str
    layer: str
    epistemic_status: str

@dataclass(frozen=True, order=True)
class Guard:
    op: str
    args: tuple
    layer: str
    origin: str
    epistemic_status: str

@dataclass(frozen=True, order=True)
class TokenSetExpr:
    op: str
    args: tuple

@dataclass(frozen=True, order=True)
class Effect:
    action: str
    component: str
    expr: TokenSetExpr

@dataclass(frozen=True, order=True)
class Restriction:
    expression: str
    origin: str

@dataclass(frozen=True, order=True)
class EvidenceRef:
    kind: str
    source: str
    detail: str

@dataclass(frozen=True, order=True)
class EditRecord:
    operator: str
    parameters: tuple
    parent_hash: str
    before_fragment: str
    after_fragment: str

@dataclass(frozen=True)
class AtomicRule:
    name: str
    variables: tuple
    context: tuple
    binders: tuple
    guards: tuple
    effects: tuple
    restrictions: tuple
    provenance: tuple
    lineage: tuple = tuple()
    edit_history: tuple = tuple()

@dataclass(frozen=True, order=True)
class BranchCondition:
    op: str
    args: tuple

@dataclass(frozen=True)
class RuleBranch:
    name: str
    condition: BranchCondition
    rule: AtomicRule
    role: str

@dataclass(frozen=True)
class ChoiceRule:
    name: str
    base_rule_hash: str
    branches: tuple
    comparison_branch: str
    policy: str
    provenance: tuple
    lineage: tuple
    edit_history: tuple

def canonical(value):
    if is_dataclass(value):
        return {
            "__type__": value.__class__.__name__,
            **{item.name: canonical(getattr(value, item.name)) for item in fields(value)},
        }
    if isinstance(value, dict):
        return {str(key): canonical(value[key]) for key in sorted(value, key=str)}
    if isinstance(value, (tuple, list)):
        return [canonical(item) for item in value]
    if isinstance(value, (set, frozenset)):
        return sorted((canonical(item) for item in value), key=lambda item: json.dumps(item, sort_keys=True))
    return value

def canonical_json(value):
    return json.dumps(canonical(value), ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def object_hash(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()

def singleton(variable):
    return TokenSetExpr("SINGLETON", (variable,))

def interval(start="p", end="n"):
    return TokenSetExpr("INTERVAL", (start, end))

def difference(left, right):
    return TokenSetExpr("DIFFERENCE", (left, right))

def input_set(name):
    return TokenSetExpr("INPUT", (name,))

def effect(component, expr):
    return Effect("ADD_MEMBERS", component, expr)

VARS = tuple(Variable(name, kind) for name, kind in (
    ("e", "Extraction"), ("s", "Token"), ("v", "Token"),
    ("p", "Token"), ("n", "Token"),
))

BASE_CONTEXT = (
    ContextAtom("nsubj(s,v)", DEPENDENCY, "EVIDENCIA_OBSERVADA_CONTEXTO"),
    ContextAtom("dep(n,v,nmod) em S009-A41", DEPENDENCY, "EVIDENCIA_OBSERVADA"),
    ContextAtom("dep(p,n,case) em S009-A41", DEPENDENCY, "EVIDENCIA_OBSERVADA_NAO_CAUSAL"),
    ContextAtom("det(d,n) em S009-A41", DEPENDENCY, "EVIDENCIA_OBSERVADA_CONTEXTO"),
    ContextAtom("p não visitado na construção de REL em S009-A41", CONSTRUCTION, "EVIDENCIA_TRACE_INSTANCIA"),
)

BASE_GUARDS = (
    Guard("ARG_ROOT_ALLOWED", ("n",), CONSTRUCTION, "Extracao1.java:1339", "EVIDENCIA_CODIGO"),
    Guard("N_CHILD_OF_V", ("n","v"), CONSTRUCTION, "caminho observado de seleção do argumento", "MODELO_RESTRITO"),
    Guard("FIRST_LEFT_CHILD", ("p","n"), CONSTRUCTION, "Sentence.java:25-30; Extracao1.java:1345-1381", "EVIDENCIA_CODIGO"),
    Guard("P_CHILD_OF_N", ("p","n"), DEPENDENCY, "estrutura de filhos do token", "DERIVADO_GRAFO"),
    Guard("P_BEFORE_N", ("p","n"), CONSTRUCTION, "Extracao1.java:1354", "EVIDENCIA_CODIGO"),
    Guard("INTERVAL_AVAILABLE", ("p","n"), CONSTRUCTION, "Extracao1.java:1363-1380", "EVIDENCIA_CODIGO"),
)

BASE_EFFECTS = (
    effect("ARG0", input_set("ARG0_BASE")),
    effect("REL", input_set("REL_BASE")),
    effect("ARG1", input_set("U")),
    effect("ARG1", interval("p","n")),
)

R_IMPL = AtomicRule(
    name="R_impl_observado_E4_S009_A41",
    variables=VARS,
    context=BASE_CONTEXT,
    binders=("p := primeiro filho à esquerda de n", "I(p,n) := intervalo inclusivo por ordem"),
    guards=BASE_GUARDS,
    effects=BASE_EFFECTS,
    restrictions=(Restriction("ordem total de tokens; intervalo inclusivo", "decisoes_implementacao_v4_e4.md"),),
    provenance=(
        EvidenceRef("TRACE", "logs/s009/reproducao_instrumentada_trace_on/e4trace.txt", "seq 18-22"),
        EvidenceRef("CODE", "src/main/java/extrai_clausulas/Extracao1.java", "linhas 1339 e 1345-1381"),
        EvidenceRef("LIMIT", "docs/proveniencia_e4_s009.md", "mecanismo comprovado apenas para S009-A41"),
    ),
)

R_IMPL_INITIAL_HASH = object_hash(R_IMPL)
print("R_IMPL", R_IMPL.name, R_IMPL_INITIAL_HASH)

R_IMPL R_impl_observado_E4_S009_A41 d4c332a43dd349ee6b7691880933432e455bbe3aa41af35814335e0ceaf4e8c4


## 3. Operadores explícitos e derivação das candidatas

Os quatro operadores científicos e a primitiva `ADD_GUARD` retornam novos
objetos. C0, C1 e C2 são derivadas independentemente de `R_impl`.

In [4]:
def _append_edit(rule, new_name, operator, parameters, before, after, **changes):
    parent_hash = object_hash(rule)
    record = EditRecord(operator, tuple(parameters), parent_hash, before, after)
    return replace(
        rule,
        name=new_name,
        lineage=rule.lineage + (parent_hash,),
        edit_history=rule.edit_history + (record,),
        **changes,
    )

def ADD_GUARD(rule, guard, new_name):
    if guard in rule.guards:
        raise ValueError("guarda duplicada")
    same_signature = [item for item in rule.guards if item.op == guard.op and item.args == guard.args]
    if same_signature:
        raise ValueError("guarda contraditória ou assinatura já ocupada")
    return _append_edit(
        rule, new_name, "ADD_GUARD", (guard.op,) + guard.args,
        "guards", canonical_json(guard), guards=rule.guards + (guard,),
    )

def ADD_GUARD_REGE(rule, new_name):
    guard = Guard(
        "STATUS_REGE_EQUALS", ("v","p",TRUE_PROVISIONAL), LINGUISTIC,
        "entrada manual provisória; parcialmente fundamentada pelo GOLD",
        "HIPOTESE_LINGUISTICA_NAO_INFERIDA",
    )
    if guard in rule.guards:
        raise ValueError("guarda de regência duplicada")
    return _append_edit(
        rule, new_name, "ADD_GUARD_REGE", ("v","p",TRUE_PROVISIONAL),
        "guards", canonical_json(guard), guards=rule.guards + (guard,),
    )

def RESTRICT_INTERVAL(rule, new_name):
    original = effect("ARG1", interval("p","n"))
    indexes = [index for index, item in enumerate(rule.effects) if item == original]
    if len(indexes) != 1:
        raise ValueError("RESTRICT_INTERVAL exige exatamente um efeito ARG1(I(p,n))")
    index = indexes[0]
    partition = (
        effect("ARG1", singleton("p")),
        effect("ARG1", difference(interval("p","n"), singleton("p"))),
    )
    effects = rule.effects[:index] + partition + rule.effects[index+1:]
    restriction = Restriction(
        "I(p,n) = {p} união (I(p,n)-{p}); partes disjuntas",
        "RESTRICT_INTERVAL",
    )
    return _append_edit(
        rule, new_name, "RESTRICT_INTERVAL", ("ARG1","I(p,n)","p"),
        canonical_json(original), canonical_json(partition),
        effects=effects, restrictions=rule.restrictions + (restriction,),
    )

def CHANGE_COMPONENT(rule, source, target, expr, new_name):
    original = effect(source, expr)
    indexes = [index for index, item in enumerate(rule.effects) if item == original]
    if len(indexes) != 1:
        raise ValueError("CHANGE_COMPONENT exige um efeito de origem explícito e único")
    replacement = effect(target, expr)
    if replacement in rule.effects:
        raise ValueError("CHANGE_COMPONENT criaria efeito duplicado")
    effects = list(rule.effects)
    effects[indexes[0]] = replacement
    return _append_edit(
        rule, new_name, "CHANGE_COMPONENT", (source,target,canonical_json(expr)),
        canonical_json(original), canonical_json(replacement), effects=tuple(effects),
    )

def SPLIT_RULE(base_rule, branches, comparison_branch, name, policy):
    if not branches:
        raise ValueError("SPLIT_RULE exige branches")
    names = [branch.name for branch in branches]
    if len(names) != len(set(names)):
        raise ValueError("nomes de branches duplicados")
    condition_keys = [canonical_json(branch.condition) for branch in branches]
    if len(condition_keys) != len(set(condition_keys)):
        raise ValueError("condições de branches sobrepostas ou duplicadas")
    if comparison_branch not in names:
        raise ValueError("branch de comparação ausente")
    statuses = {
        branch.condition.args[0]
        for branch in branches
        if branch.condition.op == "STATUS_EQUALS"
    }
    if statuses and statuses != set(REGE_STATUSES):
        raise ValueError("split por regência deve cobrir os três estados")
    c1_condition_kinds = {branch.condition.op for branch in branches}
    if "RULE_MATCH" in c1_condition_kinds or "NOT_RULE_MATCH" in c1_condition_kinds:
        if c1_condition_kinds != {"RULE_MATCH", "NOT_RULE_MATCH"}:
            raise ValueError("split por match deve ter condição e complemento")
    parent_hash = object_hash(base_rule)
    record = EditRecord("SPLIT_RULE", tuple(names), parent_hash, base_rule.name, name)
    return ChoiceRule(
        name=name,
        base_rule_hash=parent_hash,
        branches=tuple(branches),
        comparison_branch=comparison_branch,
        policy=policy,
        provenance=base_rule.provenance + (
            EvidenceRef("MODEL", "docs/protocolo_theory_revision_v4_e4.md", "split explícito com fallback"),
        ),
        lineage=(parent_hash,),
        edit_history=(record,),
    )

def derive_candidates(base_rule):
    base_hash_before = object_hash(base_rule)

    c0_partitioned = RESTRICT_INTERVAL(base_rule, "C0_PARTITIONED")
    c0 = CHANGE_COMPONENT(c0_partitioned, "ARG1", "REL", singleton("p"), "C0")

    c1_g1 = ADD_GUARD(
        base_rule,
        Guard("DEP", ("p","n","case"), DEPENDENCY, "hipótese sintática C1", "HIPOTESE"),
        "C1_G1",
    )
    c1_g2 = ADD_GUARD(
        c1_g1,
        Guard("DEP", ("n","v","nmod"), DEPENDENCY, "hipótese sintática C1", "HIPOTESE"),
        "C1_G2",
    )
    c1_partitioned = RESTRICT_INTERVAL(c1_g2, "C1_PARTITIONED")
    c1_specialized = CHANGE_COMPONENT(c1_partitioned, "ARG1", "REL", singleton("p"), "C1_TRANSFER")
    c1 = SPLIT_RULE(
        base_rule,
        (
            RuleBranch("MATCH_SYNTACTIC", BranchCondition("RULE_MATCH", (object_hash(c1_specialized),)), c1_specialized, "SPECIALIZED"),
            RuleBranch("FALLBACK", BranchCondition("NOT_RULE_MATCH", (object_hash(c1_specialized),)), base_rule, "FALLBACK"),
        ),
        "MATCH_SYNTACTIC", "C1", "exactly_one; fallback explícito",
    )

    c2_guarded = ADD_GUARD_REGE(base_rule, "C2_REGE_TRUE")
    c2_partitioned = RESTRICT_INTERVAL(c2_guarded, "C2_PARTITIONED")
    c2_specialized = CHANGE_COMPONENT(c2_partitioned, "ARG1", "REL", singleton("p"), "C2_TRANSFER")
    c2 = SPLIT_RULE(
        base_rule,
        (
            RuleBranch("REGE_TRUE", BranchCondition("STATUS_EQUALS", (TRUE_PROVISIONAL,)), c2_specialized, "SPECIALIZED"),
            RuleBranch("REGE_FALSE", BranchCondition("STATUS_EQUALS", (FALSE_PROVISIONAL,)), base_rule, "FALLBACK"),
            RuleBranch("REGE_INDETERMINATE", BranchCondition("STATUS_EQUALS", (INDETERMINATE,)), base_rule, "FALLBACK_INDETERMINATE"),
        ),
        "REGE_TRUE", "C2", "exactly_one; INDETERMINATE conservador",
    )

    assert object_hash(base_rule) == base_hash_before
    return {"C0": c0, "C1": c1, "C2": c2}

CANDIDATES = derive_candidates(R_IMPL)
assert object_hash(R_IMPL) == R_IMPL_INITIAL_HASH
assert isinstance(CANDIDATES["C0"], AtomicRule)
assert isinstance(CANDIDATES["C1"], ChoiceRule)
assert isinstance(CANDIDATES["C2"], ChoiceRule)

def specialized_rule(rule):
    if isinstance(rule, AtomicRule):
        return rule
    return next(branch.rule for branch in rule.branches if branch.name == rule.comparison_branch)

print("CANDIDATE HASHES")
for name, rule in CANDIDATES.items():
    print(name, object_hash(rule), [record.operator for record in specialized_rule(rule).edit_history],
          [record.operator for record in getattr(rule, "edit_history", tuple())])

CANDIDATE HASHES
C0 91194d427726f98be1b377d925a5790fc01fbb5596440b90d09c5341b809c427 ['RESTRICT_INTERVAL', 'CHANGE_COMPONENT'] ['RESTRICT_INTERVAL', 'CHANGE_COMPONENT']
C1 af966fec75bce1b08524e3e2d40af12258749ed255e0ad03e28753b9a5395458 ['ADD_GUARD', 'ADD_GUARD', 'RESTRICT_INTERVAL', 'CHANGE_COMPONENT'] ['SPLIT_RULE']
C2 d537c4c7088479d413ddde4cd131bdf012bb137c29bfeecdf1ffed02a01a23fe ['ADD_GUARD_REGE', 'RESTRICT_INTERVAL', 'CHANGE_COMPONENT'] ['SPLIT_RULE']


## 4. Diff e custo estruturais automáticos

O vetor é calculado das estruturas canônicas. Os pesos unitários foram
congelados no documento de decisões antes desta avaliação.

In [5]:
def item_key(item):
    return canonical_json(item)

def set_diff(left, right):
    left_map = {item_key(item): item for item in left}
    right_map = {item_key(item): item for item in right}
    removed = tuple(left_map[key] for key in sorted(set(left_map)-set(right_map)))
    added = tuple(right_map[key] for key in sorted(set(right_map)-set(left_map)))
    return added, removed

def contains_expr_op(expr, op):
    if expr.op == op:
        return True
    return any(contains_expr_op(item, op) for item in expr.args if isinstance(item, TokenSetExpr))

def atomic_diff(base, revised):
    guards_added, guards_removed = set_diff(base.guards, revised.guards)
    effects_added, effects_removed = set_diff(base.effects, revised.effects)
    restrictions_added, restrictions_removed = set_diff(base.restrictions, revised.restrictions)
    base_interval_effects = [item for item in base.effects if contains_expr_op(item.expr, "INTERVAL")]
    revised_difference_effects = [item for item in revised.effects if contains_expr_op(item.expr, "DIFFERENCE")]
    interval_edits = int(bool(base_interval_effects and revised_difference_effects))
    return {
        "guards_added": guards_added,
        "guards_removed": guards_removed,
        "effects_added": effects_added,
        "effects_removed": effects_removed,
        "interval_edits": interval_edits,
        "restrictions_added": restrictions_added,
        "restrictions_removed": restrictions_removed,
        "splits": 0,
    }

def rule_diff(base, candidate):
    revised = specialized_rule(candidate)
    result = atomic_diff(base, revised)
    result["splits"] = int(isinstance(candidate, ChoiceRule))
    return result

UNIT_WEIGHTS = {
    "guard": 1,
    "effect": 1,
    "interval": 1,
    "restriction": 1,
    "split": 1,
}

def diff_vector(diff):
    return {
        "guards_added": len(diff["guards_added"]),
        "guards_removed": len(diff["guards_removed"]),
        "effects_added": len(diff["effects_added"]),
        "effects_removed": len(diff["effects_removed"]),
        "interval_edits": diff["interval_edits"],
        "restrictions_added": len(diff["restrictions_added"]),
        "restrictions_removed": len(diff["restrictions_removed"]),
        "splits": diff["splits"],
    }

def edit_cost(vector):
    return (
        UNIT_WEIGHTS["guard"] * (vector["guards_added"] + vector["guards_removed"])
        + UNIT_WEIGHTS["effect"] * (vector["effects_added"] + vector["effects_removed"])
        + UNIT_WEIGHTS["interval"] * vector["interval_edits"]
        + UNIT_WEIGHTS["restriction"] * (vector["restrictions_added"] + vector["restrictions_removed"])
        + UNIT_WEIGHTS["split"] * vector["splits"]
    )

def reconcile_history(base, candidate, diff):
    revised = specialized_rule(candidate)
    operators = tuple(record.operator for record in revised.edit_history)
    vector = diff_vector(diff)
    guard_ops = sum(operator in {"ADD_GUARD", "ADD_GUARD_REGE"} for operator in operators)
    assert vector["guards_added"] == guard_ops
    assert ("RESTRICT_INTERVAL" in operators) == (vector["interval_edits"] == 1)
    assert ("CHANGE_COMPONENT" in operators) == any(
        item.component == "REL" and item.expr == singleton("p") for item in diff["effects_added"]
    )
    assert vector["splits"] == int(isinstance(candidate, ChoiceRule))
    assert object_hash(base) == R_IMPL_INITIAL_HASH
    return True

DIFFS = {name: rule_diff(R_IMPL, rule) for name, rule in CANDIDATES.items()}
EDIT_SUMMARIES = {}
for name, diff in DIFFS.items():
    assert reconcile_history(R_IMPL, CANDIDATES[name], diff)
    vector = diff_vector(diff)
    EDIT_SUMMARIES[name] = {"vector": vector, "unit_cost": edit_cost(vector)}

print(json.dumps({"weights": UNIT_WEIGHTS, "candidates": EDIT_SUMMARIES}, ensure_ascii=False, indent=2))

{
  "weights": {
    "guard": 1,
    "effect": 1,
    "interval": 1,
    "restriction": 1,
    "split": 1
  },
  "candidates": {
    "C0": {
      "vector": {
        "guards_added": 0,
        "guards_removed": 0,
        "effects_added": 2,
        "effects_removed": 1,
        "interval_edits": 1,
        "restrictions_added": 1,
        "restrictions_removed": 0,
        "splits": 0
      },
      "unit_cost": 5
    },
    "C1": {
      "vector": {
        "guards_added": 2,
        "guards_removed": 0,
        "effects_added": 2,
        "effects_removed": 1,
        "interval_edits": 1,
        "restrictions_added": 1,
        "restrictions_removed": 0,
        "splits": 1
      },
      "unit_cost": 8
    },
    "C2": {
      "vector": {
        "guards_added": 1,
        "guards_removed": 0,
        "effects_added": 2,
        "effects_removed": 1,
        "interval_edits": 1,
        "restrictions_added": 1,
        "restrictions_removed": 0,
        "splits": 1
      },
     

## 5. Matcher e executor genéricos

O matcher deriva `p` como o primeiro filho à esquerda de `n` e produz
`I(p,n)`. O executor avalia as expressões formais da regra selecionada. O
fallback é determinado pelo tipo/branches da regra, nunca pelo nome C0/C1/C2.

In [6]:
def add_edge(graph, source, target, key, relation, layer):
    graph.add_edge(source, target, key=key, rel=relation, layer=layer)

def build_input_graph(case):
    graph = nx.MultiDiGraph(name=f"G_base_{case.case_id}")
    graph.add_node("ROOT", type="Root")
    graph.add_node("e", type="Extraction", case_id=case.case_id)
    graph.add_node("construction", type="ConstructionContext")
    for token_id, data in PARSES[case.sentence_id].items():
        graph.add_node(token_id, type="Token", **data)
    for token_id, data in PARSES[case.sentence_id].items():
        head = data["head"] if data["head"] != 0 else "ROOT"
        add_edge(graph, token_id, head, f"dep:{token_id}", data["deprel"], DEPENDENCY)
    add_edge(graph, "construction", case.relation_core, "core", "CANDIDATE_REL_CORE", CONSTRUCTION)
    for index, target_input in enumerate(case.targets):
        add_edge(graph, "construction", target_input.n, f"argroot:{index}", "ARG_ROOT_INPUT", CONSTRUCTION)
    return graph

def ordered_token_ids(graph):
    ids = [node for node, data in graph.nodes(data=True) if data.get("type") == "Token"]
    return tuple(sorted(ids, key=lambda node: graph.nodes[node]["ordem"]))

def dependency_children(graph, head):
    children = []
    for child, target, key, data in graph.in_edges(head, keys=True, data=True):
        if target == head and data.get("layer") == DEPENDENCY:
            children.append(child)
    return tuple(sorted(children, key=lambda node: graph.nodes[node]["ordem"]))

def dependency_exists(graph, child, head, relation):
    edge_data = graph.get_edge_data(child, head, default={})
    return any(data.get("layer") == DEPENDENCY and data.get("rel") == relation for data in edge_data.values())

def interval_by_order(graph, start, end):
    if start not in graph or end not in graph:
        return tuple()
    start_order = graph.nodes[start]["ordem"]
    end_order = graph.nodes[end]["ordem"]
    if start_order > end_order:
        return tuple()
    return tuple(
        token_id for token_id in ordered_token_ids(graph)
        if start_order <= graph.nodes[token_id]["ordem"] <= end_order
    )

@dataclass(frozen=True)
class MatchEnvironment:
    bindings: tuple
    interval: tuple
    status_rege: str
    status_origin: str
    status_uses_gold: bool
    evidence: tuple

@dataclass(frozen=True)
class MatchResult:
    rule_name: str
    matched: bool
    guard_results: tuple
    environment: MatchEnvironment

def env_bindings(env):
    return dict(env.bindings)

def derive_environment(graph, case, target_input):
    n = target_input.n
    v = case.relation_core
    n_order = graph.nodes[n]["ordem"]
    left_children = tuple(
        child for child in dependency_children(graph, n)
        if graph.nodes[child]["ordem"] < n_order
    )
    p = left_children[0] if left_children else None
    current_interval = interval_by_order(graph, p, n) if p is not None else tuple()
    bindings = (("e","e"), ("v",v), ("n",n), ("p",p))
    evidence = (
        ("children_in_order", dependency_children(graph,n)),
        ("left_children", left_children),
        ("derived_p", p),
        ("derived_interval", current_interval),
    )
    return MatchEnvironment(
        bindings, current_interval, target_input.status_rege,
        target_input.status_origin, target_input.status_uses_gold, evidence,
    )

def evaluate_guard(guard, graph, env):
    bindings = env_bindings(env)
    resolve = lambda name: bindings.get(name, name)
    if guard.op == "ARG_ROOT_ALLOWED":
        n = resolve(guard.args[0])
        return n in graph and graph.nodes[n].get("deprel") in ARG_ROOT_ALLOWED
    if guard.op == "N_CHILD_OF_V":
        n, v = map(resolve, guard.args)
        return n in graph and graph.nodes[n].get("head") == v
    if guard.op == "FIRST_LEFT_CHILD":
        p, n = map(resolve, guard.args)
        if p is None or n not in graph:
            return False
        left = [child for child in dependency_children(graph,n) if graph.nodes[child]["ordem"] < graph.nodes[n]["ordem"]]
        return bool(left) and p == left[0]
    if guard.op == "P_CHILD_OF_N":
        p, n = map(resolve, guard.args)
        return p is not None and p in graph and graph.nodes[p].get("head") == n
    if guard.op == "P_BEFORE_N":
        p, n = map(resolve, guard.args)
        return p is not None and graph.nodes[p]["ordem"] < graph.nodes[n]["ordem"]
    if guard.op == "INTERVAL_AVAILABLE":
        return bool(env.interval)
    if guard.op == "DEP":
        child, head, relation = guard.args
        return dependency_exists(graph, resolve(child), resolve(head), relation)
    if guard.op == "STATUS_REGE_EQUALS":
        expected = guard.args[2]
        return env.status_rege == expected
    raise ValueError(f"guarda desconhecida: {guard.op}")

def match_atomic(rule, graph, env):
    results = tuple((guard, evaluate_guard(guard, graph, env)) for guard in rule.guards)
    return MatchResult(rule.name, all(value for _, value in results), results, env)

def find_rule_by_hash(choice, target_hash):
    matches = [branch.rule for branch in choice.branches if object_hash(branch.rule) == target_hash]
    if not matches:
        raise ValueError("regra referenciada pelo branch não encontrada")
    return matches[0]

def evaluate_branch_condition(condition, choice, graph, env):
    if condition.op in {"RULE_MATCH", "NOT_RULE_MATCH"}:
        referenced = find_rule_by_hash(choice, condition.args[0])
        matched = match_atomic(referenced, graph, env).matched
        return matched if condition.op == "RULE_MATCH" else not matched
    if condition.op == "STATUS_EQUALS":
        return env.status_rege == condition.args[0]
    raise ValueError(f"condição de branch desconhecida: {condition.op}")

def select_choice_branch(choice, graph, env):
    evaluations = tuple(
        (branch, evaluate_branch_condition(branch.condition, choice, graph, env))
        for branch in choice.branches
    )
    selected = [branch for branch, value in evaluations if value]
    if len(selected) != 1:
        raise ValueError(f"ChoiceRule não exclusivo: {choice.name}; selecionados={len(selected)}")
    return selected[0], tuple((branch.name, value) for branch, value in evaluations)

def eval_token_expr(expr, graph, env, case, evaluation_trace):
    bindings = env_bindings(env)
    if expr.op == "INPUT":
        source = expr.args[0]
        mapping = {"ARG0_BASE": case.arg0_base, "REL_BASE": case.rel_base, "U": case.u}
        result = tuple(mapping[source])
    elif expr.op == "SINGLETON":
        value = bindings[expr.args[0]]
        result = (value,)
    elif expr.op == "INTERVAL":
        start, end = (bindings[name] for name in expr.args)
        result = interval_by_order(graph, start, end)
        assert result == env.interval
    elif expr.op == "DIFFERENCE":
        left = eval_token_expr(expr.args[0], graph, env, case, evaluation_trace)
        right = set(eval_token_expr(expr.args[1], graph, env, case, evaluation_trace))
        result = tuple(item for item in left if item not in right)
    else:
        raise ValueError(f"expressão desconhecida: {expr.op}")
    evaluation_trace.append((expr, result))
    return result

def apply_atomic_effects(rule, graph, env, case):
    state = {"ARG0": set(), "REL": set(), "ARG1": set()}
    evaluation_trace = []
    for current_effect in rule.effects:
        assert current_effect.action == "ADD_MEMBERS"
        members = eval_token_expr(current_effect.expr, graph, env, case, evaluation_trace)
        state[current_effect.component].update(members)
    return state, tuple(evaluation_trace)

def merge_state(target_state, contribution):
    for component, members in contribution.items():
        target_state[component].update(members)

def ordered_state(graph, state):
    def key(token_id):
        return graph.nodes[token_id].get("ordem", -1)
    return {component: tuple(sorted(members, key=key)) for component, members in state.items()}

def apply_candidate(candidate, base_rule, case, graph):
    state = {"ARG0": set(), "REL": set(), "ARG1": set()}
    target_records = []
    for target_input in case.targets:
        env = derive_environment(graph, case, target_input)
        if isinstance(candidate, AtomicRule):
            candidate_match = match_atomic(candidate, graph, env)
            if candidate_match.matched:
                selected_rule = candidate
                branch_name = "ATOMIC_MATCH"
                branch_evaluations = (("ATOMIC_MATCH", True),)
            else:
                selected_rule = base_rule
                branch_name = "NO_MATCH_IDENTITY_FALLBACK"
                branch_evaluations = (("ATOMIC_MATCH", False), ("IDENTITY_FALLBACK", True))
        else:
            branch, branch_evaluations = select_choice_branch(candidate, graph, env)
            selected_rule = branch.rule
            branch_name = branch.name
            candidate_match = match_atomic(selected_rule, graph, env)
        contribution, expression_trace = apply_atomic_effects(selected_rule, graph, env, case)
        merge_state(state, contribution)
        target_records.append({
            "n": target_input.n,
            "p": env_bindings(env)["p"],
            "interval": env.interval,
            "status_rege": env.status_rege,
            "status_origin": env.status_origin,
            "status_uses_gold": env.status_uses_gold,
            "branch": branch_name,
            "branch_evaluations": branch_evaluations,
            "selected_rule": selected_rule.name,
            "selected_rule_hash": object_hash(selected_rule),
            "guard_results": tuple((guard.op, value) for guard, value in candidate_match.guard_results),
            "expression_trace": expression_trace,
        })
    return ordered_state(graph, state), tuple(target_records)

## 6. Testes estruturais e fixtures

Fixtures são testes de infraestrutura, não exemplos de corpus. Esta seção
cobre imutabilidade, pré-condições, exclusividade, `INDETERMINATE`, `U` não
vazio, pontuação, IDs não contíguos e ausência de GOLD na construção.

In [7]:
ASSERTIONS = []

def record_assertion(name, condition):
    assert condition, name
    ASSERTIONS.append(name)

record_assertion("R_IMPL_IMMUTABLE", object_hash(R_IMPL) == R_IMPL_INITIAL_HASH)
record_assertion("CANDIDATES_COMPLETE_RULE_OBJECTS", set(CANDIDATES) == {"C0","C1","C2"})
record_assertion("INDEPENDENT_LINEAGE", all(R_IMPL_INITIAL_HASH in specialized_rule(rule).lineage for rule in CANDIDATES.values()))
record_assertion("DIFF_HISTORY_RECONCILED", all(reconcile_history(R_IMPL, CANDIDATES[name], DIFFS[name]) for name in CANDIDATES))
record_assertion("UNIT_COSTS_FROZEN", set(UNIT_WEIGHTS.values()) == {1})

def raises_value_error(function):
    try:
        function()
    except ValueError:
        return True
    return False

record_assertion(
    "CHANGE_COMPONENT_PRECONDITION",
    raises_value_error(lambda: CHANGE_COMPONENT(R_IMPL, "ARG1", "REL", singleton("p"), "INVALID")),
)
record_assertion(
    "DUPLICATE_GUARD_REJECTED",
    raises_value_error(lambda: ADD_GUARD(R_IMPL, R_IMPL.guards[0], "INVALID")),
)
duplicate_branch = RuleBranch("B1", BranchCondition("STATUS_EQUALS", (TRUE_PROVISIONAL,)), R_IMPL, "FALLBACK")
record_assertion(
    "OVERLAPPING_BRANCHES_REJECTED",
    raises_value_error(lambda: SPLIT_RULE(
        R_IMPL,
        (duplicate_branch, RuleBranch("B2", duplicate_branch.condition, R_IMPL, "FALLBACK")),
        "B1", "INVALID", "invalid",
    )),
)

fixture_graph = nx.MultiDiGraph(name="fixture_non_contiguous_with_punctuation")
fixture_graph.add_node(10, type="Token", id=10, form="para", pos="ADP", head=30, deprel="case", ordem=1)
fixture_graph.add_node(20, type="Token", id=20, form=",", pos=".", head=30, deprel="punct", ordem=2)
fixture_graph.add_node(30, type="Token", id=30, form="área", pos="NOUN", head=40, deprel="nmod", ordem=3)
record_assertion("INTERVAL_PRESERVES_PUNCTUATION", interval_by_order(fixture_graph,10,30) == (10,20,30))
record_assertion("INTERVAL_SUPPORTS_NONCONTIGUOUS_IDS", interval_by_order(fixture_graph,10,30) != tuple(range(10,31)))

base_case = next(case for case in CASES if case.case_id == "S009-A41")
indeterminate_target = replace(base_case.targets[0], status_rege=INDETERMINATE,
                               status_origin="fixture sintético; não corpus", status_uses_gold=False,
                               justification="teste do fallback conservador")
indeterminate_case = replace(base_case, case_id="FIXTURE-INDETERMINATE", targets=(indeterminate_target,))
indeterminate_graph = build_input_graph(indeterminate_case)
indeterminate_state, indeterminate_records = apply_candidate(CANDIDATES["C2"], R_IMPL, indeterminate_case, indeterminate_graph)
record_assertion("INDETERMINATE_FALLBACK", indeterminate_state == {"ARG0":(1,), "REL":(2,), "ARG1":(3,4,5)})
record_assertion("INDETERMINATE_BRANCH_EXPLICIT", indeterminate_records[0]["branch"] == "REGE_INDETERMINATE")

u_case = replace(base_case, case_id="FIXTURE-U-NONEMPTY", arg1_baseline=(3,4,5,6), u=(6,))
u_graph = build_input_graph(u_case)
u_state, u_records = apply_candidate(CANDIDATES["C2"], R_IMPL, u_case, u_graph)
record_assertion("COMPOSITE_ARG1_U_PRESERVED", u_state == {"ARG0":(1,), "REL":(2,3), "ARG1":(4,5,6)})

record_assertion("NO_GOLD_FIELD_IN_CONSTRUCTION_CASE", not any("gold" in field.name.lower() for field in fields(ConstructionCase)))
record_assertion("NO_GOLD_ARGUMENT_IN_MATCHER", "gold" not in inspect.signature(derive_environment).parameters)
record_assertion("NO_GOLD_ARGUMENT_IN_EXECUTOR", "gold" not in inspect.signature(apply_candidate).parameters)

print("STRUCTURAL/FIXTURE ASSERTIONS", len(ASSERTIONS), "PASS")

STRUCTURAL/FIXTURE ASSERTIONS 16 PASS


## 7. Avaliação nos oito casos reais

As candidatas congeladas são aplicadas aos oito casos rastreados. A avaliação
consulta os oráculos somente depois da construção.

In [8]:
def baseline_state(case):
    return {"ARG0":case.arg0_base, "REL":case.rel_base, "ARG1":case.arg1_baseline}

def expected_state(oracle):
    return {"ARG0":oracle.expected_e4_arg0, "REL":oracle.expected_e4_rel, "ARG1":oracle.expected_e4_arg1}

def gold_state(oracle):
    return {"ARG0":oracle.gold_arg0, "REL":oracle.gold_rel, "ARG1":oracle.gold_arg1}

def state_diff(left, right):
    return {
        component: {
            "removed": tuple(item for item in left[component] if item not in right[component]),
            "added": tuple(item for item in right[component] if item not in left[component]),
        }
        for component in ("ARG0","REL","ARG1")
    }

def state_text(case, state):
    return {component: token_text(case.sentence_id, ids) for component, ids in state.items()}

def canonical_graph_hash(graph):
    nodes = sorted((str(node), canonical(data)) for node, data in graph.nodes(data=True))
    edges = sorted((str(u),str(v),str(key),canonical(data)) for u,v,key,data in graph.edges(keys=True,data=True))
    return hashlib.sha256(json.dumps({"nodes":nodes,"edges":edges}, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

RESULTS = []
for candidate_name, candidate in CANDIDATES.items():
    for case in CASES:
        oracle = ORACLES[case.case_id]
        graph = build_input_graph(case)
        result, target_records = apply_candidate(candidate, R_IMPL, case, graph)
        expected = expected_state(oracle) if case.kind.startswith("E_MINUS") else baseline_state(case)
        if result == expected:
            classification = "PASS_EXPLORATORY"
        elif case.kind.startswith("E_MINUS"):
            classification = "FAIL_CORRECTION"
        else:
            classification = "FAIL_OVERAPPLICATION"
        derived_pairs = tuple((record["n"],record["p"]) for record in target_records)
        assert derived_pairs == oracle.expected_p_by_n
        reconstructed_baseline, baseline_records = apply_candidate(R_IMPL, R_IMPL, case, graph)
        assert reconstructed_baseline == baseline_state(case)
        assert all(any(expr.op == "INTERVAL" for expr, _ in record["expression_trace"])
                   for record in target_records)
        RESULTS.append({
            "candidate": candidate_name,
            "case_id": case.case_id,
            "kind": case.kind,
            "graph_hash": canonical_graph_hash(graph),
            "rule_hash": object_hash(candidate),
            "result": result,
            "result_text": state_text(case,result),
            "baseline": baseline_state(case),
            "expected_e4": expected_state(oracle),
            "gold": gold_state(oracle),
            "diff_from_baseline": state_diff(baseline_state(case),result),
            "diff_from_gold": state_diff(gold_state(oracle),result),
            "gold_exact_by_boundary": result == gold_state(oracle),
            "targets": target_records,
            "classification": classification,
            "edit_summary": EDIT_SUMMARIES[candidate_name],
            "notes": case.notes,
        })

def get_result(candidate, case_id):
    return next(item for item in RESULTS if item["candidate"] == candidate and item["case_id"] == case_id)

priority = {"FAIL_OVERAPPLICATION":3, "FAIL_CORRECTION":2, "INCONCLUSIVE":1, "PASS_EXPLORATORY":0}
GLOBAL_RESULTS = {}
for candidate_name in CANDIDATES:
    labels = [item["classification"] for item in RESULTS if item["candidate"] == candidate_name]
    GLOBAL_RESULTS[candidate_name] = max(labels, key=lambda label: priority[label])

print("TEST MATRIX")
for item in RESULTS:
    transferred = tuple(
        record["p"] for record in item["targets"]
        if record["selected_rule"] != R_IMPL.name and record["branch"] not in {"FALLBACK","REGE_FALSE","REGE_INDETERMINATE","NO_MATCH_IDENTITY_FALLBACK"}
    )
    print(item["candidate"], item["case_id"], item["classification"],
          "transferred=",transferred, "result=",item["result_text"])
print("GLOBAL", GLOBAL_RESULTS)

TEST MATRIX
C0 S009-A41 PASS_EXPLORATORY transferred= (3,) result= {'ARG0': 'Catalano', 'REL': 'voltou para', 'ARG1': 'a área'}
C0 S009-A42 PASS_EXPLORATORY transferred= (3,) result= {'ARG0': 'Catalano', 'REL': 'voltou para', 'ARG1': 'a área depois de viver em Londres'}
C0 S009-A43 PASS_EXPLORATORY transferred= (3,) result= {'ARG0': 'Catalano', 'REL': 'voltou para', 'ARG1': 'a área depois de viver em Londres e em Glasgow , onde estudou design de jóias'}
C0 S009-A44 PASS_EXPLORATORY transferred= (3,) result= {'ARG0': 'Catalano', 'REL': 'voltou para', 'ARG1': 'a área depois de viver em Londres e em Glasgow , onde estudou design de jóias en a Glasgow School of Art'}
C0 S005-A28 FAIL_OVERAPPLICATION transferred= (18,) result= {'ARG0': 'alguém', 'REL': 'tem para', 'ARG1': 'dignidade continuar a protestar contra esse fato'}
C0 S010-A51 FAIL_OVERAPPLICATION transferred= (6,) result= {'ARG0': 'Eu', 'REL': 'visitei en', 'ARG1': 'os espetáculos universitários a Universidade de Northampton'}
C0 S

## 8. Asserções finais e registro do resultado

As conclusões são limitadas ao modelo e ao conjunto fechado. Não há E+ E4
confirmado nem reconhecimento operacional de regência.

In [9]:
record_assertion("EIGHT_REAL_CASES", len(CASES) == 8)
record_assertion("TWENTY_FOUR_REAL_APPLICATIONS", len(RESULTS) == 24)
record_assertion("BASELINE_RECONSTRUCTED_FROM_U_AND_INTERVALS", True)
record_assertion("MATCHER_INTERVAL_USED_BY_EXECUTOR", all(
    all(any(expr.op == "INTERVAL" for expr, _ in record["expression_trace"]) for record in item["targets"])
    for item in RESULTS
))

for candidate_name in CANDIDATES:
    record_assertion(
        f"{candidate_name}_CORRECTS_S009_A41",
        get_result(candidate_name,"S009-A41")["result"] == {"ARG0":(1,),"REL":(2,3),"ARG1":(4,5)},
    )

record_assertion("GLOBAL_RESULTS_REPRODUCE_CLOSED_SET", GLOBAL_RESULTS == {
    "C0":"FAIL_OVERAPPLICATION", "C1":"FAIL_OVERAPPLICATION", "C2":"PASS_EXPLORATORY",
})
record_assertion("C0_OVERAPPLIES_S005", get_result("C0","S005-A28")["classification"] == "FAIL_OVERAPPLICATION")
record_assertion("C1_PRESERVES_S005", get_result("C1","S005-A28")["classification"] == "PASS_EXPLORATORY")
record_assertion("C1_OVERAPPLIES_S010", get_result("C1","S010-A51")["classification"] == "FAIL_OVERAPPLICATION")
record_assertion("C2_PRESERVES_MODELED_CONTROLS", all(
    get_result("C2",case.case_id)["classification"] == "PASS_EXPLORATORY"
    for case in CASES if case.kind == "CONTROL"
))
record_assertion("C2_STATUS_SOURCE_RECORDED", all(
    record["status_origin"] and isinstance(record["status_uses_gold"], bool)
    for item in RESULTS if item["candidate"] == "C2" for record in item["targets"]
))
record_assertion("R_IMPL_STILL_IMMUTABLE_AFTER_EVALUATION", object_hash(R_IMPL) == R_IMPL_INITIAL_HASH)

MODELED_CONTROL_OUTPUTS = 4
TOTAL_CONTROL_BASELINE_OUTPUTS = 15
UNMODELED_CONTROL_OUTPUTS = TOTAL_CONTROL_BASELINE_OUTPUTS - MODELED_CONTROL_OUTPUTS
record_assertion("NON_REGRESSION_COVERAGE_LIMIT_RECORDED", UNMODELED_CONTROL_OUTPUTS == 11)

FINAL_RECORD = {
    "protocol_commit": PROTOCOL_COMMIT,
    "decisions_commit": DECISIONS_COMMIT,
    "input_integrity": "PASS",
    "r_impl_hash": R_IMPL_INITIAL_HASH,
    "candidate_hashes": {name:object_hash(rule) for name,rule in CANDIDATES.items()},
    "candidate_edits": EDIT_SUMMARIES,
    "real_cases": [case.case_id for case in CASES],
    "real_applications": len(RESULTS),
    "assertions_passed": len(ASSERTIONS),
    "global_results": GLOBAL_RESULTS,
    "indeterminate_fixture": "PASS_CONSERVATIVE_FALLBACK",
    "u_nonempty_fixture": "PASS_PRESERVED",
    "full_15_non_regression": "INCONCLUSIVE_NOT_FULLY_MODELED",
    "e4_positive_examples": "NONE_CONFIRMED",
    "rege_recognizer": "NOT_IMPLEMENTED_RESEARCH_QUESTION",
    "interpretation": "formal rule-editing result; not Java revision or generalization",
}
print(json.dumps(FINAL_RECORD, ensure_ascii=False, indent=2))

{
  "protocol_commit": "4183b5ecddee1f8e42d82f5d21de2381513384b6",
  "decisions_commit": "54a65af0e2023eb23683f1e020123ac692401c98",
  "input_integrity": "PASS",
  "r_impl_hash": "d4c332a43dd349ee6b7691880933432e455bbe3aa41af35814335e0ceaf4e8c4",
  "candidate_hashes": {
    "C0": "91194d427726f98be1b377d925a5790fc01fbb5596440b90d09c5341b809c427",
    "C1": "af966fec75bce1b08524e3e2d40af12258749ed255e0ad03e28753b9a5395458",
    "C2": "d537c4c7088479d413ddde4cd131bdf012bb137c29bfeecdf1ffed02a01a23fe"
  },
  "candidate_edits": {
    "C0": {
      "vector": {
        "guards_added": 0,
        "guards_removed": 0,
        "effects_added": 2,
        "effects_removed": 1,
        "interval_edits": 1,
        "restrictions_added": 1,
        "restrictions_removed": 0,
        "splits": 0
      },
      "unit_cost": 5
    },
    "C1": {
      "vector": {
        "guards_added": 2,
        "guards_removed": 0,
        "effects_added": 2,
        "effects_removed": 1,
        "interval_edits": 